## 🔧 3. Knowledge Retreival

**Goal**: Add tool to fetch information from a file / search index

- `FileSearchTool` — RAG over uploaded documents (server-side)

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential
from agent_framework import tool

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    FileSearchTool,
    WebSearchTool,
    FunctionTool as AzureFunctionTool,
)
from agent_framework.foundry import FoundryAgent

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

print(f"📡 Project endpoint: {PROJECT_ENDPOINT}")
print(f"🧠 Model: {MODEL}")
print("✅ Connected to Foundry")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


📡 Project endpoint: https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model: gpt-4o
✅ Connected to Foundry


### File Search (RAG) 📂
Uploaded product docs into a Vector Store, register an agent with `FileSearchTool` in Foundry, then query via `FoundryAgent`.

In [2]:
# ── B2. Register File Search agent in Foundry ────────────────────────────────
FILE_AGENT_NAME = "ProductSpecialist-af"
# Vector store - DemoProductInfoIndex
vectorstoredid = "vs_ZK0XW6cotxNUoZ8rBAqWShcW"

file_agent_version = project.agents.create_version(
    agent_name=FILE_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Contoso product specialist. Use the file search tool to "
            "look up product details, pricing, and policies from the company "
            "knowledge base. Always cite the source."
        ),
        tools=[FileSearchTool(vector_store_ids=[vectorstoredid])],
    ),
)

# Connect via Agent Framework
file_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=file_agent_version.name,
    agent_version=str(file_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {file_agent_version.name} v{file_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(file_agent).__name__}")

2026-10-26 11:16:11,490 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
✅ Agent registered in Foundry: ProductSpecialist-af v1
🤖 FoundryAgent connected: FoundryAgent


In [3]:
# ── B3. Ask questions grounded in the uploaded doc ───────────────────────────
for question in [
    "What is the price of the Contoso Widget Pro?",
    "Compare Widget Pro vs Widget Lite.",
    "What is the return policy for hardware?",
]:
    resp = await file_agent.run(question)
    print(f"\n❓ {question}")
    print(f"🗨️  {resp}")

print(f"\n🔍 Check Foundry portal → Tracing to see these interactions!")


❓ What is the price of the Contoso Widget Pro?
🗨️  The price of the Contoso Widget Pro is $49.99.

❓ Compare Widget Pro vs Widget Lite.
🗨️  Here's a comparison between **Widget Pro** and **Widget Lite**:

### Widget Pro
- **Price**: $49.99
- **Description**: Premium, titanium-reinforced housing with smart-connect technology. Suitable for enterprise deployments.
- **Warranty**: 3 years
- **Weight**: 250g

### Widget Lite
- **Price**: $19.99
- **Description**: Lightweight and budget-friendly, designed for personal and small-team use. Easy setup in under 5 minutes.
- **Warranty**: 1 year
- **Weight**: 120g

**Key Differences**:
- **Price**: Widget Pro is more expensive, reflecting its more robust features and build.
- **Build Quality**: The Pro variant offers a more durable construction compared to the Lite variant.
- **Target Audience**: Pro is geared towards enterprises, while Lite is aimed at individuals or small teams.
- **Weight and Portability**: Widget Lite is lighter, making it m

---
### Cleanup 🧹

In [4]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
# Delete agents from Foundry
for name in [FILE_AGENT_NAME]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🧹 Deleted agent: {name}")
    except Exception as e:
        print(f"⚠️  Could not delete {name}: {e}")

print("✅ Cleanup complete")

🧹 Deleted agent: ProductSpecialist-af
✅ Cleanup complete


---
### ✅ What you demonstrated
| Capability | Foundry Definition | Agent Framework Feature |
|---|---|---|
| Document Q&A (RAG) | `FileSearchTool(vector_store_ids=[...])` | Runs server-side via `FoundryAgent` |
| Server-side tracing | All interactions visible in Foundry portal → Tracing | |

**Next →** [04_agent_tools_mcp.ipynb](./04_agent_tools_mcp.ipynb) — Agent with MCP tools.